In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd
import random
import string

def generate_noisy_data(word: str, num_distractors: int = 49) -> tuple[str, str]:
    reversed_word = word.upper()[::-1]
    distractors = []
    for _ in range(num_distractors):
        length = random.randint(5, 8)
        distractor = ''.join(random.choices(string.ascii_letters + string.digits, k=length))
        distractors.append(distractor)

    all_items = distractors + [reversed_word]
    random.shuffle(all_items)
    data_block_str = ", ".join(all_items)
    return data_block_str, word

@kbench.task(name="find_reversed_word_single", store_task=False)
def find_reversed_word_single(llm, data_block: str, answer: str) -> bool:
    prompt = f"""Below is a jumbled sequence of 50 random alphanumeric strings. Within this data, there is one English word spelled backwards.

Data Block: [{data_block}]

Task: Find the English word hidden in the sequence, reverse it to its original spelling, and output only that word."""

    response = llm.prompt(prompt)
    return answer.lower() in response.lower()

@kbench.task(name="find_reversed_word_in_noise")
def find_reversed_word_in_noise(llm) -> float:
    words_to_find = [
        "COMPUTER", "PYTHON", "KAGGLE", "BENCHMARK", "EVALUATE",
        "SEQUENCE", "LIBRARY", "FRAMEWORK", "ANALYSIS", "INSIGHT"
    ]

    eval_data = []
    for word in words_to_find:
        data_block, answer = generate_noisy_data(word)
        eval_data.append({"data_block": data_block, "answer": answer})

    df = pd.DataFrame(eval_data)

    with kbench.client.enable_cache():
        runs = find_reversed_word_single.evaluate(
            llm=[llm],
            evaluation_data=df,
            n_jobs=2,
            remove_run_files=True
        )

    eval_df = runs.as_dataframe()
    if eval_df.empty or 'result' not in eval_df.columns:
        return 0.0

    accuracy = float(eval_df.result.mean())
    return accuracy

find_reversed_word_in_noise.run(kbench.llm)